In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [2]:
load_dotenv()

True

In [3]:

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.7
)

"""model = ChatGroq(
    model="llama-3.1-8b-instant",
    max_tokens=50
)"""

"""from langchain_openai import ChatOpenAI
import os


model = ChatOpenAI(
    openai_api_key=os.environ.get("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="nvidia/nemotron-3-ultra-550b-a55b:free" # The ':free' tag is the secret password!
)""";


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [4]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [5]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).text

    # update the answer in the state
    state['answer'] = answer

    return state

In [6]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [7]:
# execute

intial_state = {'question': 'How far is India from becoming the Vishwa Guru? give me the definite answer in 2 to 3 words... and provide its reasoning '}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The answer is: **Subjective and ongoing.**

**Reasoning:**
The concept of "Vishwa Guru" (Global Teacher) is not a quantifiable metric, but rather a soft-power aspiration. Whether India is close to achieving this status depends on which criteria are prioritized:

*   **Arguments for progress:** India’s growing economy, leadership in digital public infrastructure, success in space exploration (Chandrayaan-3), and an increasingly assertive diplomatic stance on the global stage suggest it is gaining significant international influence.
*   **Arguments for distance:** Critics argue that to be a true "Vishwa Guru," India must first address internal systemic challenges, such as poverty alleviation, human development indices (education and healthcare quality), and the preservation of social cohesion. 

Ultimately, the term is a vision rather than a destination with a fixed finish line; its attainment depends on India's ability to balance its economic rise with its commitment to democratic valu

In [8]:
model.invoke('How far is moon from the earth?').text

'Because the Moon’s orbit around the Earth is elliptical (oval-shaped) rather than a perfect circle, the distance between them is constantly changing.\n\nHere is the breakdown:\n\n*   **Average distance:** About **238,855 miles** (384,400 kilometers).\n*   **Perigee (closest point):** About **225,623 miles** (363,104 kilometers).\n*   **Apogee (farthest point):** About **252,088 miles** (405,696 kilometers).\n\n**To put that distance into perspective:**\n*   You could fit all the other planets in our solar system side-by-side in the space between the Earth and the Moon.\n*   It takes light about **1.3 seconds** to travel from the Moon to the Earth.\n*   If you could drive a car to the Moon at a constant 60 mph (97 km/h), it would take you about **six months** to get there.'